# Isopropylamine

In [1]:
using Clapeyron, Metaheuristics, Printf

In [2]:
like_parameter = """Clapeyron Database File
PCSAFT Like Parameters [csvtype = like,grouptype = PCSAFT]
species,Mw,segment,sigma,epsilon,n_H,n_e
isopropylamine,59.11,2.68622,3.440089484,228.4542964,1,1
"""

assoc_parameter = """Clapeyron Database File
PCSAFT Assoc Parameters [csvtype = assoc,grouptype = PCSAFT]
species1,site1,species2,site2,epsilon_assoc,bondvol
isopropylamine,H,isopropylamine,e,800.0,0.05
"""

model = PCSAFT(["isopropylamine"], userlocations = [like_parameter, assoc_parameter])

print(model.params.epsilon_assoc.values)

Clapeyron.Compressed4DMatrix{Float64, Vector{Float64}}[800.0]

In [3]:
# Run this ONCE to fix your CSV files
function fix_line_endings(filename)
    content = read(filename, String)
    fixed = replace(content, "\r\n" => "\n")
    write(filename, fixed)
    println("Fixed: $filename")
end

fix_line_endings("satp_isopropylamine.csv")
fix_line_endings("rhol_isopropylamine.csv")

Fixed: satp_isopropylamine.csv
Fixed: rhol_isopropylamine.csv


In [4]:
### Saturation pressure — output in Pa (SI)
function my_saturation_p(model::EoSModel, T::Float64)
    sat = saturation_pressure(model, T)
    return sat[1]   # Pa
end

# Liquid density — output in kg/m³
function my_liquid_density(model::EoSModel, T::Float64)
    sat  = saturation_pressure(model, T)
    Mw   = model.params.Mw[1]      # g/mol
    rhol = 1.0 / sat[2]            # mol/m³
    return rhol * Mw / 1000.0      # mol/m³ × g/mol ÷ 1000 = kg/m³
end

println(my_liquid_density(model, 293.15))
println(my_saturation_p(model, 273.15))

690.4363331927423
21377.730312375086


In [11]:
toestimate = [
    Dict(
        :param => :segment,
        :lower => 1.0,
        :upper => 10.0,
        :guess => 3.
    ),
    Dict(
        :param => :epsilon,
        :lower => 100.,
        :upper => 500.,
        :guess => 250.
    ),
    Dict(
        :param => :sigma,
        :factor => 1e-10,
        :lower => 2.0,
        :upper => 7.0,
        :guess => 5.
    ),
    Dict(
        :param   => :epsilon_assoc,
        :indices => (1,1),
        :lower   => 100.0,
        :upper   => 1000.0,
        :guess   => 750.0
    ),
    Dict(
        :param   => :bondvol,
        :indices => (1,1),
        :lower   => 0.001,
        :upper   => 1.0,
        :guess   => 0.05
    )
]

5-element Vector{Dict{Symbol, Any}}:
 Dict(:upper => 10.0, :param => :segment, :guess => 3.0, :lower => 1.0)
 Dict(:upper => 500.0, :param => :epsilon, :guess => 250.0, :lower => 100.0)
 Dict(:factor => 1.0e-10, :param => :sigma, :upper => 7.0, :guess => 5.0, :lower => 2.0)
 Dict(:upper => 1000.0, :param => :epsilon_assoc, :indices => (1, 1), :guess => 750.0, :lower => 100.0)
 Dict(:upper => 1.0, :param => :bondvol, :indices => (1, 1), :guess => 0.05, :lower => 0.001)

In [12]:
estimator, objective, x0, upper, lower = Estimation(
    model,
    toestimate,
    [
        "rhol_isopropylamine.csv",
        "satp_isopropylamine.csv",
    ]
)
 
println("Initial objective value: ", objective(x0))

Initial objective value: 1.4070551396669035


In [13]:
method = ECA(; options = Options(iterations = 10000000, seed = 999))
 
params_opt, model_opt = optimize(objective, estimator, method)

([2.7005983234802304, 225.64992553959664, 3.427268076379271, 843.8870263193186, 0.03474194071430654], PCSAFT{BasicIdeal, Float64}("isopropylamine"))

In [8]:
using CSV, DataFrames, Printf

function calculate_AAD(model, csv_file, property_func)
    df = CSV.read(csv_file, DataFrame, comment="#", skipto=4)
    
    input_col  = names(df)[1]          # first column = input (T)
    output_col = names(df)[2]          # second column = out_xxx (experimental)
    
    inputs   = df[!, input_col]
    exp_vals = df[!, output_col]
    
    println("\n=== AAD: $csv_file ===")
    @printf("%-10s  %-12s  %-12s  %-8s\n", input_col, "exp", "calc", "ARD%")
    
    errors = Float64[]
    for (i, x) in enumerate(inputs)
        calc = property_func(model, x)
        err  = abs(calc - exp_vals[i]) / abs(exp_vals[i]) * 100
        push!(errors, err)
        @printf("%-10.4f  %-12.6f  %-12.6f  %-8.4f\n", x, exp_vals[i], calc, err)
    end
    
    aard = sum(errors) / length(errors)
    @printf("AARD = %.4f%%\n", aard)
    return aard
end

calculate_AAD (generic function with 1 method)

In [9]:
aard_p   = calculate_AAD(model_opt, "satp_isopropylamine.csv",   my_saturation_p)

LoadError: UndefVarError: `model_opt` not defined

In [10]:
aard_p   = calculate_AAD(model_opt, "rhol_isopropylamine.csv",   my_liquid_density)

LoadError: UndefVarError: `model_opt` not defined